# A Deep-Genetic Algorithm (Deep-GA) Approach for High-Dimensional Nonlinear Parabolic Partial Differential Equations

## For Google Colaboratory and Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0,'/content/drive/MyDrive/Colab Notebooks/DeepGA')

!pip install munch

Mounted at /content/drive


## Import Library

In [2]:
import os
import time
import json
import munch
import logging

import numpy as np
import pandas as pd
import tensorflow as tf

import equation as eqn
from modelDeepGeneticAlgorithm import BSDESolver

## Parameters for the Genetic Algorithm

In [3]:
start_time = time.time()

# minimum and maximum values of u_0
min_value = 0
max_value = 100 # 100 for pricing_default_risk_d100, 10 for hjb_lq_d100
range_value = max_value - min_value

n = 10 # number of chromosomes in population
p_c = 0.8 # probability of crossover
p_m = 0.4 # probability of mutation
g = 10 # number of generations
num_iter = 1 # number of iterations for training the model in each fitness function call

## Parameters for BSDESolver

In [4]:
# The path to load json file
a = '/content/drive/MyDrive/Colab Notebooks/DeepGA/pricing_default_risk_d100.json'
#a = '/content/drive/MyDrive/Colab Notebooks/DeepGA/hjb_lq_d100.json'

with open(a) as json_data_file:
    config = json.load(json_data_file)
config = munch.munchify(config)
bsde = getattr(eqn, config.eqn_config.eqn_name)(config.eqn_config)
tf.keras.backend.set_floatx(config.net_config.dtype)

y_init = range_value/2
config.net_config.y_init_range = [y_init, y_init]

model = BSDESolver(config, bsde)
valid_data = model.get_valid_data()
loss = model.loss_ga(valid_data)

model.show_model()

Model: "nonshared_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 feed_forward_sub_net (FeedF  multiple                 35880     
 orwardSubNet)                                                   
                                                                 
 feed_forward_sub_net_1 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_2 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_3 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_4 (Fee  multiple             

## The Genetic Algorithm

In [5]:
def createPopulation():
    a = range(1, n+1)
    b = np.array(a)
    c = b * range_value / n
    pop = c
    pop = pd.DataFrame(pop)
    pop.columns = ['u_0']
    
    return pop

def fitness(pop):
    fitness = []
    valid_data = model.get_valid_data()

    for i in range(len(pop)):
        a = pop['u_0'][i]
        y_init = [a]
        model.set_y_init(y_init)
        
        model.train(num_iter)

    for i in range(len(pop)):
        a = pop['u_0'][i]
        y_init = [a]
        model.set_y_init(y_init)

        loss = model.loss_ga(valid_data)
        
        fitness.append(loss)
    
    pop['fitness'] = fitness
    
    return pop

def randomSelection():
    position = np.random.permutation(n)
    
    return position[0], position[1]

def crossover(pop):
    popc = []
    
    for i in range(len(pop)):
        if np.random.rand() <= p_c:
            a, b = randomSelection()
            popc.append((pop.loc[a][0] + pop.loc[b][0])/2)
            
    popc = pd.DataFrame(popc)
    popc.columns = ['u_0']
    
    return popc

def mutation(pop):
    popm = []
    
    for i in range(len(pop)):
        if np.random.rand() <= p_m:
            popm.append(pop.loc[i][0] + np.random.rand()*(max_value/50) - (max_value/100))
    
    if len(popm) == 0:
        popm.append(pop.loc[i][0] + np.random.rand()*(max_value/50) - (max_value/100))
    
    popm = pd.DataFrame(popm)
    popm.columns = ['u_0']
    
    return popm

def combinePopulation(pop1, pop2, pop3):
    popAll = pop1.copy()
    popAll = popAll.append(pop2)
    popAll = popAll.append(pop3)
    popAll = popAll.drop_duplicates()
    popAll.index = range(len(popAll))

    return popAll

def sort(pop):
    pop = pop.sort_values(by=['fitness'])
    pop.index = range(len(pop))

    return pop

def elimination(pop):
    pop = pop.head(n)
    
    return pop

def GA():
    pop1 = createPopulation()
    pop2 = createPopulation()
    pop3 = createPopulation()
    
    print('Generation 0')
    print('Average u_0 in population 1 : ' + str(np.average(pop1['u_0'])))
    print('Average u_0 in population 2 : ' + str(np.average(pop2['u_0'])))
    print('Average u_0 in population 3 : ' + str(np.average(pop3['u_0'])))
    print('===================================================================')
    print()
    
    for i in range(0, g):
        popc1 = crossover(pop1)
        popc2 = crossover(pop2)
        popc3 = crossover(pop3)
        
        popm1 = mutation(pop1)
        popm2 = mutation(pop2)
        popm3 = mutation(pop3)
        
        popAll1 = combinePopulation(pop1, popc1, popm1)
        popAll2 = combinePopulation(pop2, popc2, popm2)
        popAll3 = combinePopulation(pop3, popc3, popm3)
        
        f = 2.0/100
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])+range_value*f
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])-range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])+range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])-range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])+range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])-range_value*f
        
        f = 4.0/100
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])+range_value*f
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])-range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])+range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])-range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])+range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])-range_value*f
        
        popAll1 = fitness(popAll1)
        popAll2 = fitness(popAll2)
        popAll3 = fitness(popAll3)
        
        popAll1 = sort(popAll1)
        popAll2 = sort(popAll2)
        popAll3 = sort(popAll3)
        
        pop1 = elimination(popAll1)
        pop2 = elimination(popAll2)
        pop3 = elimination(popAll3)
        
        print('pop1')
        print(pop1)
        print()
        print('pop2')
        print(pop2)
        print()
        print('pop3')
        print(pop3)
        print()

        pop1.drop(['fitness'], axis=1)
        pop2.drop(['fitness'], axis=1)
        pop3.drop(['fitness'], axis=1)
        
        print('Generation ' + str(i+1))
        print('Average u_0 in population 1 : ' + str(np.average(pop1['u_0'])))
        print('Average u_0 in population 2 : ' + str(np.average(pop2['u_0'])))
        print('Average u_0 in population 3 : ' + str(np.average(pop3['u_0'])))
        print('===================================================================')
        print()
    
    pop = combinePopulation(pop1, pop2, pop3)
    
    return pop

## Result

In [6]:
pop = GA()

u_0 = np.average(pop['u_0'])
print('Final u_0 : ' + str(u_0))

elapsed_time = time.time() - start_time
print('Total time : ' + str(elapsed_time))

Generation 0
Average u_0 in population 1 : 55.0
Average u_0 in population 2 : 55.0
Average u_0 in population 3 : 55.0

pop1
         u_0     fitness
0  57.000000   26.210250
1  59.000000   28.573927
2  55.000000   30.197909
3  60.000000   32.070538
4  53.000000   40.757641
5  51.000000   58.117757
6  50.528233   63.229450
7  50.000000   69.421748
8  65.000000   71.673480
9  70.000000  150.043363

pop2
         u_0     fitness
0  57.000000   26.156601
1  59.000000   28.283333
2  55.000000   30.384114
3  60.000000   31.662547
4  53.000000   41.186665
5  51.000000   58.792620
6  50.000000   70.221116
7  65.000000   70.688993
8  69.934511  147.032804
9  70.000000  148.500184

pop3
         u_0     fitness
0  57.000000   25.113749
1  59.000000   27.957259
2  60.000000   31.691951
3  53.000000   38.686548
4  51.000000   55.551765
5  50.000000   66.607235
6  65.000000   72.467271
7  70.000000  151.988012
8  45.000000  158.670044
9  70.960442  175.169089

Generation 1
Average u_0 in population